In [ ]:
#####
# import weigthed ranking coeff
import sys
project_paths = ["./ranking_correlation"]
for project_path in project_paths:
    if not (project_path in sys.path):
        sys.path.append(project_path)

from WA_params import *
from WA_utils import Metrics, M_calculator, Dataset, beta_par_calculator


def O_completer(O, P, N):
    """Derives TN and FN from the total positives/negatives and the known TP/FP"""
    O.update({'TN': N - O['FP'], 'FN': P - O['TP']})
    return O


# indeces_pred_pos = rng_utils.choice(range(D.N_tot), N_pred_pos, replace=False)
def indeces_pred_pos_calculator(O, indeces_pos, indeces_neg):
    """Randomly samples TP indices from positives and FP indices from negatives to form predicted positives"""
    tp_list = list(rng_utils.choice(indeces_pos, O['TP'], replace=False))
    fp_list = list(rng_utils.choice(indeces_neg, O['FP'], replace=False))
    return np.array(sorted(tp_list + fp_list))


rng_utils = np.random.default_rng(142)


## Goal

The goal of this notebook is calculating the additional cost caused by using the wrong evaluation metric.

Consider a validation setting in which classifier hyperparameters %(or a decision threshold) 
must be selected. 
To emulate model performance across different hyperparameter configurations, we assume a parametric relationship between the True Positive Rate (TPR, i.e., $\mathrm{TP}/\mathrm{P}$) and the False Positive Rate (FPR, i.e., $\mathrm{FP}/\mathrm{N}$), namely  $\mathrm{TPR} = (\mathrm{FPR})^2$, which corresponds to an area under the ROC curve of $2/3$.
The candidate models are generated from this FPR-TPR relationship, by uniformely sampling FPR in the interval [0,1] with step size $0.01$.

We analyze four representative scenarios:
* mild class imbalance ($r_+=0.2$) with strongly asymmetric costs favoring false positives ($r_C=0.01$);
* mild imbalance ($r_+=0.2$) with strongly asymmetric costs favoring false negatives ($r_C=0.99$);
* strong imbalance ($r_+=0.01$) with higher cost associated with false negatives ($r_C=0.9$);
* strong imbalance ($r_+=0.01$) with higher cost associated with false positives ($r_C=0.1$).

For each scenario, we select, from the previously described set of candidates, the optimal model according to each evaluation metric $X$ under consideration.
We then compute the performance gap $\Delta \mathrm{TCC}$, defined as the difference between the TCC of the model selected by metric $X$ and the minimum achievable TCC among all candidate models.
Therefore, $\Delta \mathrm{TCC}$ quantifies the economical cost induced by the sub-optimal classifier performance, due to the use of metric $X$ during model selection.



In [ ]:
def O_to_df_UCC(O):
    """Builds a per-instance cost dataframe from a confusion matrix outcome, assigning C_FP and C_FN costs"""
    P = O['TP'] + O['FN']
    N = O['TN'] + O['FP']
    N_tot = N + P

    indeces_pos = rng_utils.choice(range(N_tot), P, replace=False)
    indeces_neg = np.array([x for x in range(N_tot) if x not in indeces_pos])

    M = M_calculator(C_frac, use_case='churn')
    C_FP_list = [M] * (N + P)
    C_FN_list = [max(0, R * P_eff - M) for R in R_list[:N_tot]]

    df_UCC = pd.DataFrame({'C_FP': C_FP_list, 'C_FN': C_FN_list})
    df_UCC['is_plus'] = [True if i in indeces_pos else False for i in range(N_tot) ]
    
    indeces_pred_pos = indeces_pred_pos_calculator(O=O, indeces_pos=indeces_pos, indeces_neg=indeces_neg)
    df_UCC['is_pred_plus'] = [True if i in indeces_pred_pos else False for i in range(N_tot) ]
    return df_UCC

In [ ]:
def Outcome_examiner(O, Metric):
    """Computes all evaluation metrics and TCC for a single confusion matrix outcome, appending results to Metric"""
    df_UCC = O_to_df_UCC(O)                
    cost_FN_list = df_UCC[df_UCC.is_plus & pd.Series([not x for x in df_UCC.is_pred_plus])]['C_FN']
    cost_FP_list = df_UCC[df_UCC.is_pred_plus & pd.Series([not x for x in df_UCC.is_plus])]['C_FP']
    
    TP = O['TP']
    FN = O['FN']
    TN = O['TN']
    FP = O['FP']
    P = TP + FN
    N = TN + FP
   
    D = Dataset(P=P, N=N)
    D.set_cost(C_FN=C_frac / (1 - C_frac), C_FP=1)

    TCC = sum(cost_FN_list) + sum(cost_FP_list)

    D.set_confusion_matrix(FN=FN, FP=FP)
    Metric.append_to_list('precision', D.prec)
    Metric.append_to_list('recall', D.rec)
    Metric.append_to_list('specificity', D.spec)
    Metric.append_to_list('NPV', D.NPV)
    Metric.append_to_list('accuracy', D.accuracy)
    Metric.append_to_list('F1', D.F1)
    Metric.append_to_list('kappa', D.kappa)
    Metric.append_to_list('jaccard', D.jaccard)
    Metric.append_to_list('P4', D.P4)
    Metric.append_to_list('ROC-AUC', D.ROC_AUC)
    Metric.append_to_list('CBA', D.CBA)
    Metric.append_to_list('IAM', D.IAM)
    Metric.append_to_list('MCC', D.MCC)
    Metric.append_to_list('informedness', D.informedness)
    Metric.append_to_list('markedness', D.markedness)
    Metric.append_to_list('ACD', D.ACD) 
    Metric.append_to_list('B-ROC', D.B_ROC)
    Metric.append_to_list('WRA', D.WRA)
    D.set_w()
    Metric.append_to_list('WCA', D.WCA)
    Metric.append_to_list('WA', D.WA)
    Metric.append_to_list('H', D.get_H_complete())
    
    alpha, beta = beta_par_calculator(df_UCC=df_UCC)
    Metric.append_to_list('H informed', D.get_H_complete(alpha=alpha, beta=beta))
    Metric.append_to_list('EWA', D.get_EWA(alpha=alpha, beta=beta))
    
    Metric.append_to_list('G-mean', D.G_mean)
    Metric.append_to_list('TCC', TCC)
    Metric.metrics_dic['outcome']= Metric.metrics_dic.get('outcome',[]) + [str(O)]
    return Metric
    


In [ ]:
def O_list_examiner(O_list, dic_results):
    """For each metric, finds the outcome it selects as best and records its TCC gap from the optimal"""
    Metric = Metrics()    
        
    for O in O_list:
        Metric = Outcome_examiner(O, Metric)
    
    TCC_lst = Metric.metrics_dic['TCC']
    TCC_min = min(TCC_lst)
        
    for metric in metrics_of_interests:
        lst = Metric.metrics_dic[metric]
        if metric in ('ACD', 'C-score'):
            lst = [-x for x in lst]
    
        max_index = max(range(len(lst)), key=lambda i: lst[i])
    
        dic_results[metric] = dic_results.get(metric, []) + [TCC_lst[max_index]-TCC_min]

    return dic_results

In [ ]:
def single_scenario_sampler(r_P, C_frac, N_tot, n_samples, dic_results, tpr='square'):
    """Runs n_samples repetitions of O_list_examiner for a given scenario and aggregates mean TCC gaps per metric"""
    P = int(round(r_P * N_tot))
    N = N_tot - P
    dic_results_single_scenario = {}
    if tpr == 'square':
        O_list = [O_completer(O={'TP': int(round(rho**2 * P)), 'FP': int(round(rho * N))}, P=P, N=N) for rho in np.arange(0, 1, 0.01)]
    elif tpr == 'linear':
        O_list = [O_completer(O={'TP': int(round(rho * P)), 'FP': int(round(rho * N))}, P=P, N=N) for rho in np.arange(0, 1, 0.01)]
    # elif tpr == '9-power':
    #     O_list = [O_completer(O={'TP': int(round(rho**9 * P)), 'FP': int(round(rho * N))}, P=P, N=N) for rho in np.arange(0, 1, 0.01)]
    for i in range(n_samples):
        dic_results_single_scenario = O_list_examiner(O_list=O_list, dic_results=dic_results_single_scenario)
    for metric in dic_results_single_scenario.keys():
        dic_results[metric] = dic_results.get(metric, []) + [round(np.mean(dic_results_single_scenario[metric]))]
    return dic_results


In [6]:
N_tot = 200
use_case = 'churn'
n_samples = 1000
    
dic_results = {}

# scenario 1
r_P = 0.2
C_frac = 0.01 
dic_results = single_scenario_sampler(r_P, C_frac, N_tot, n_samples, dic_results)

# # scenario 2
r_P = 0.2
C_frac = 0.99
dic_results = single_scenario_sampler(r_P, C_frac, N_tot, n_samples, dic_results)

# scenario 3
r_P = 0.01
C_frac = 0.9
dic_results = single_scenario_sampler(r_P, C_frac, N_tot, n_samples, dic_results)

# scenario 4
r_P = 0.01
C_frac = 0.1
dic_results = single_scenario_sampler(r_P, C_frac, N_tot, n_samples, dic_results)

df = pd.DataFrame(dic_results)

In [7]:
df[df.columns[:20]]

,accuracy,CBA,IAM,H,WCA,kappa,informedness,ROC-AUC,WRA,MCC,markedness,precision,NPV,P4,G-mean,jaccard,F1,recall,B-ROC,specificity
0,0,795,795,0,0,0,0,0,0,0,3842,3916,0,2778,2778,3916,3916,3916,3916,0
1,591,565,565,591,1,591,591,591,591,591,17,1,591,292,292,1,1,1,1,591
2,7,7,7,7,253,253,253,253,253,253,253,253,253,253,154,253,253,253,253,7
3,0,0,0,0,0,3155,3155,3155,3155,3155,3155,3155,3155,3155,1853,3155,3155,3155,3155,0
